# Advanced Problems with Solutions — Class Construction and Metaclasses

In this notebook we are going to continue exploring how Python classes are actually constructed.

The main topics are still the same:

- class body execution
- class namespaces
- `exec`
- `type`
- metaclasses
- `__prepare__`
- `__new__`
- `__init__`
- metaclass `__call__`
- descriptors and class creation
- inheritance and metaclass selection

But this time we will work through a set of **advanced problems**.

The important difference is that we will not jump directly to large finished solutions.

Instead, each problem will be broken into small steps.

We will inspect intermediate objects, namespaces and method resolution behavior as we go.


The goal is not just to make the final code work.

The goal is to understand **why** it works.

That is especially important with metaprogramming, because a small amount of code can trigger quite a lot of Python machinery behind the scenes.


In [1]:
import inspect
from collections import OrderedDict
from functools import wraps


We will use a few `assert` statements throughout the notebook.

They are not a replacement for a full test suite, but they are very useful here because they let us verify each step immediately.


# Problem 1 — Build a Class from a Generated Namespace

Suppose we want to generate a class called `Counter`.

The final class should behave like this:

```python
counter = Counter(10)
counter.increment()
counter.increment(5)
counter.value
```

and the final value should be `16`.

But we are **not** going to start with a normal `class Counter:` statement.


Let's first think about what Python needs in order to create a class using `type`.

We need three things:

1. a class name
2. a tuple of base classes
3. a class namespace


Let's start with just the class name.


In [2]:
class_name = "Counter"


For this class we do not need any special base class.

So we can use an empty tuple.

Python will make the class inherit from `object`.


In [3]:
class_bases = ()


Now we need the class namespace.

We could build that dictionary manually.

But for this problem we are going to generate the namespace by executing a class-like body with `exec`.


In [4]:
class_body = '''
def __init__(self, start=0):
    self.value = start

def increment(self, amount=1):
    self.value += amount
    return self.value
'''


At this point `class_body` is only a string.

No function objects have been created yet.

Let's verify that.


In [5]:
type(class_body), class_body[:30]


(str, '\ndef __init__(self, start=0):\n')

Now let's create an empty dictionary.

This dictionary will become the local namespace in which the code is executed.


In [6]:
class_namespace = {}


Now execute the body.


In [7]:
exec(class_body, globals(), class_namespace)


The important thing to inspect now is the dictionary.

Before the call to `exec` it was empty.

After the call, it should contain actual function objects.


In [8]:
class_namespace


{'__init__': <function __main__.__init__(self, start=0)>,
 'increment': <function __main__.increment(self, amount=1)>}

Let's inspect the types of the two generated values.


In [9]:
type(class_namespace["__init__"]), type(class_namespace["increment"])


(function, function)

So `exec` has transformed the function definitions in our string into real Python function objects and placed them in our namespace.


We now have the three ingredients needed by `type`.


In [10]:
Counter = type(class_name, class_bases, class_namespace)


Let's inspect the result.


In [11]:
Counter, type(Counter), Counter.__bases__


(__main__.Counter, type, (object,))

And now let's use the class normally.


In [12]:
counter = Counter(10)
counter.increment()
counter.increment(5)
counter.value


16

Let's verify the final state.


In [13]:
assert counter.value == 16
assert type(Counter) is type


### What did we learn?

The important point is that the class statement is not magic.

At a conceptual level, Python needs a namespace containing the class attributes and methods, and then it needs a class constructor — normally `type` — to turn that namespace into a class object.


# Problem 2 — A Subtle `exec` Name-Resolution Problem

Now let's look at a more difficult case.

Suppose we generate this function dynamically:

```python
factor = 10

def multiply(x):
    return x * factor
```

We might expect that if both names are created by the same `exec`, the function will always be able to find `factor`.

But the exact mappings passed to `exec` matter.


We will intentionally create a problematic version first.


In [14]:
problem_globals = {"__builtins__": __builtins__}
problem_locals = {}


Now let's execute the code using **different** dictionaries for globals and locals.


In [15]:
source = '''
factor = 10

def multiply(x):
    return x * factor
'''

exec(source, problem_globals, problem_locals)


Let's inspect where the two names were stored.


In [16]:
problem_locals.keys(), problem_globals.keys()


(dict_keys(['factor', 'multiply']), dict_keys(['__builtins__']))

Both `factor` and `multiply` were assigned to the local mapping.

But functions do not keep a reference to the local mapping used by `exec`.

A function keeps a reference to its **global namespace**.


In [17]:
problem_locals["multiply"].__globals__ is problem_globals


True

So when `multiply` later looks for `factor`, it looks in `problem_globals`.

But `factor` is in `problem_locals`.

Let's see the consequence.


In [18]:
try:
    problem_locals["multiply"](3)
except NameError as exc:
    print(type(exc).__name__ + ":", exc)


NameError: name 'factor' is not defined


This is one of those cases where understanding namespaces matters much more than memorizing syntax.


A simple solution for dynamically generated module-like code is to use the **same dictionary** for globals and locals.


In [19]:
shared_namespace = {"__builtins__": __builtins__}
exec(source, shared_namespace, shared_namespace)


Now the generated function's global namespace is the same dictionary that contains `factor`.


In [20]:
shared_namespace["multiply"].__globals__ is shared_namespace


True

And the call works.


In [21]:
shared_namespace["multiply"](3)


30

In [22]:
assert shared_namespace["multiply"](3) == 30


### General lesson

When generated functions need to resolve names that were also dynamically generated, pay close attention to the globals mapping captured by those functions.


# Problem 3 — Generate Several Methods Correctly

Let's dynamically create a class with methods:

- `power_2`
- `power_3`
- `power_4`
- `power_5`

For example:

```python
ops.power_3(2)
```

should return `8`.


This problem looks simple, but it contains a classic closure trap.


Let's first write the incorrect version.


In [23]:
bad_namespace = {}

for exponent in range(2, 6):
    def power(self, value):
        return value ** exponent

    bad_namespace[f"power_{exponent}"] = power


Now let's build the class.


In [24]:
BadPowerOps = type("BadPowerOps", (), bad_namespace)
bad_ops = BadPowerOps()


What do you think the following calls will return?


In [25]:
[
    bad_ops.power_2(2),
    bad_ops.power_3(2),
    bad_ops.power_4(2),
    bad_ops.power_5(2),
]


[32, 32, 32, 32]

All of the functions refer to the same outer variable `exponent`.

They do not each remember the value that `exponent` had when the function was created.

By the time the methods are called, the loop has finished and `exponent` is `5`.


We need each generated function to close over a different value.

A small function factory is a clean solution.


In [26]:
def make_power(exponent):
    def power(self, value):
        return value ** exponent

    return power


Now let's build a fresh namespace.


In [27]:
power_namespace = {}

for exponent in range(2, 6):
    power_namespace[f"power_{exponent}"] = make_power(exponent)


Let's inspect the keys before creating the class.


In [28]:
power_namespace.keys()


dict_keys(['power_2', 'power_3', 'power_4', 'power_5'])

Now create the class.


In [29]:
PowerOps = type("PowerOps", (), power_namespace)
ops = PowerOps()


And test the methods one by one.


In [30]:
ops.power_2(3)


9

In [31]:
ops.power_3(2)


8

In [32]:
ops.power_4(2)


16

In [33]:
ops.power_5(2)


32

In [34]:
assert ops.power_2(3) == 9
assert ops.power_3(2) == 8
assert ops.power_4(2) == 16
assert ops.power_5(2) == 32


### Why this matters for metaprogramming

When classes are generated in loops, closures are often generated in loops too.

That means normal Python closure rules still apply.

Dynamic class construction does not create a special exception to those rules.


# Problem 4 — Observe the Class Namespace While the Body Executes

So far we have mostly looked at the namespace **after** class-body code finished executing.

Now let's make the namespace itself observable.

We are going to create a custom mapping that logs every assignment made by the class body.


Remember that a metaclass can define `__prepare__`.

That method runs before the class body is executed.

Its return value becomes the namespace used while the class body runs.


First, let's create a dictionary subclass that prints every assignment.


In [35]:
class LoggingNamespace(dict):
    def __setitem__(self, key, value):
        print(f"assigning {key!r}")
        super().__setitem__(key, value)


Now we need a metaclass that returns this mapping from `__prepare__`.


In [36]:
class LoggingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases):
        print(f"preparing namespace for {name}")
        return LoggingNamespace()


Notice that we have not overridden `__new__` yet.

That is fine.

The inherited implementation from `type` can still construct the class.


Now let's define a class using the metaclass.

Watch the output carefully.


In [37]:
class TracedClass(metaclass=LoggingMeta):
    x = 10
    y = 20

    def add(self):
        return self.x + self.y


preparing namespace for TracedClass
assigning '__module__'
assigning '__qualname__'
assigning '__firstlineno__'
assigning 'x'
assigning 'y'
assigning 'add'
assigning '__static_attributes__'


The output shows assignments that Python makes automatically as well as assignments from our class body.

For example, you should see names such as:

- `__module__`
- `__qualname__`
- `x`
- `y`
- `add`


Now let's inspect the finished class.


In [38]:
TracedClass.__dict__


mappingproxy({'__module__': '__main__',
              '__firstlineno__': 1,
              'x': 10,
              'y': 20,
              'add': <function __main__.TracedClass.add(self)>,
              '__static_attributes__': (),
              '__dict__': <attribute '__dict__' of 'TracedClass' objects>,
              '__weakref__': <attribute '__weakref__' of 'TracedClass' objects>,
              '__doc__': None})

And use it normally.


In [39]:
obj = TracedClass()
obj.add()


30

In [40]:
assert obj.add() == 30


### Key idea

`__prepare__` lets the metaclass influence the environment in which the class body executes.

That means metaclasses can observe or control class-body assignments **before the class object exists**.


# Problem 5 — Reject Duplicate Class Attributes

Let's build on the previous problem.

Suppose we want Python to reject this:

```python
class Example:
    x = 10
    x = 20
```

Normally the second assignment simply replaces the first one.

But with a custom class namespace, we can detect the duplicate during execution of the class body.


We will create a dictionary that refuses to overwrite existing user-defined names.


In [41]:
class NoDuplicateNamespace(dict):
    _special_names = {
        "__module__",
        "__qualname__",
        "__annotations__",
        "__classcell__",
    }

    def __setitem__(self, key, value):
        if key in self and key not in self._special_names:
            raise TypeError(f"duplicate class attribute: {key}")

        super().__setitem__(key, value)


Now let's create the metaclass.


In [42]:
class NoDuplicateMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases):
        return NoDuplicateNamespace()


A valid class should still work normally.


In [43]:
class ValidSettings(metaclass=NoDuplicateMeta):
    host = "localhost"
    port = 8000


In [44]:
ValidSettings.host, ValidSettings.port


('localhost', 8000)

Now let's try a duplicate.


In [45]:
try:
    class InvalidSettings(metaclass=NoDuplicateMeta):
        host = "localhost"
        host = "example.com"
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: duplicate class attribute: host


The error happens while the class body is being executed.

The class object is never successfully created.


This is a good example of why `__prepare__` is more than just an obscure metaclass hook.

It can change the semantics of the class body itself.


# Problem 6 — Automatically Add a Class Identifier

Now let's move from `__prepare__` to metaclass `__new__`.

Suppose every class created by our metaclass should automatically receive a class attribute called `qualified_id`.

For a class called `Invoice`, we want something like:

```python
Invoice.qualified_id == "Invoice"
```

For a subclass called `PaidInvoice`, we want:

```python
PaidInvoice.qualified_id == "Invoice.PaidInvoice"
```


This means the value depends on the class being created and on its base classes.

That makes metaclass `__new__` a natural place to compute it.


In [46]:
class IdentifierMeta(type):
    def __new__(mcls, name, bases, namespace):
        parent_ids = [
            getattr(base, "qualified_id")
            for base in bases
            if hasattr(base, "qualified_id")
        ]

        if parent_ids:
            qualified_id = parent_ids[0] + "." + name
        else:
            qualified_id = name

        namespace["qualified_id"] = qualified_id

        return super().__new__(mcls, name, bases, namespace)


Let's create a root class.


In [47]:
class Invoice(metaclass=IdentifierMeta):
    pass


In [48]:
Invoice.qualified_id


'Invoice'

Now a subclass.


In [49]:
class PaidInvoice(Invoice):
    pass


In [50]:
PaidInvoice.qualified_id


'Invoice.PaidInvoice'

And another level.


In [51]:
class ArchivedPaidInvoice(PaidInvoice):
    pass


In [52]:
ArchivedPaidInvoice.qualified_id


'Invoice.PaidInvoice.ArchivedPaidInvoice'

In [53]:
assert Invoice.qualified_id == "Invoice"
assert PaidInvoice.qualified_id == "Invoice.PaidInvoice"
assert ArchivedPaidInvoice.qualified_id == "Invoice.PaidInvoice.ArchivedPaidInvoice"


### Where did the attribute come from?

It was not written in any of the class bodies.

The metaclass inserted it into the namespace before delegating to `type.__new__`.


# Problem 7 — Validate a Class at Creation Time

Imagine we are creating command classes.

Every concrete command must define:

- a non-empty string `command_name`
- a callable method `execute`

We want invalid command classes to fail immediately when Python tries to create them.


Let's first create a metaclass that will perform the validation.


In [54]:
class CommandMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        if name == "Command":
            return cls

        command_name = namespace.get("command_name")
        execute = namespace.get("execute")

        if not isinstance(command_name, str) or not command_name.strip():
            raise TypeError(
                f"{name} must define a non-empty string command_name"
            )

        if not callable(execute):
            raise TypeError(
                f"{name} must define execute(self, ...)"
            )

        return cls


Now the root class can be created without satisfying those requirements.


In [55]:
class Command(metaclass=CommandMeta):
    pass


Let's define a valid command.


In [56]:
class HelloCommand(Command):
    command_name = "hello"

    def execute(self, person):
        return f"Hello, {person}!"


In [57]:
HelloCommand().execute("Ada")


'Hello, Ada!'

In [58]:
assert HelloCommand().execute("Ada") == "Hello, Ada!"


Now let's try a class that forgets `command_name`.


In [59]:
try:
    class MissingNameCommand(Command):
        def execute(self):
            return None
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: MissingNameCommand must define a non-empty string command_name


And one that provides a non-callable `execute`.


In [60]:
try:
    class BrokenExecuteCommand(Command):
        command_name = "broken"
        execute = 123
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: BrokenExecuteCommand must define execute(self, ...)


### Why validate at class creation time?

Because the error is structural.

There is no reason to wait until someone creates an instance.

If the class definition itself violates the framework contract, failing early makes the problem much easier to find.


# Problem 8 — Automatic Registration of Subclasses

Let's make our command system more useful.

Instead of manually maintaining a dictionary such as:

```python
commands = {
    "hello": HelloCommand,
    ...
}
```

we want the metaclass to register command classes automatically.


We will use a registry stored on the metaclass itself.


In [61]:
class RegistryMeta(type):
    registry = {}

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)

        if name != "RegisteredCommand":
            command_name = namespace.get("command_name")

            if not isinstance(command_name, str) or not command_name:
                raise TypeError(
                    f"{name} must define command_name"
                )

            if command_name in mcls.registry:
                raise ValueError(
                    f"duplicate command_name: {command_name}"
                )

            mcls.registry[command_name] = cls

        return cls


Now let's define the root class.


In [62]:
class RegisteredCommand(metaclass=RegistryMeta):
    @classmethod
    def create(cls, command_name, *args, **kwargs):
        command_cls = RegistryMeta.registry[command_name]
        return command_cls(*args, **kwargs)


The registry is currently empty.


In [63]:
RegistryMeta.registry


{}

Now create one subclass.


In [64]:
class AddCommand(RegisteredCommand):
    command_name = "add"

    def execute(self, a, b):
        return a + b


The class should have registered itself during class construction.


In [65]:
RegistryMeta.registry


{'add': __main__.AddCommand}

Let's add another class.


In [66]:
class MultiplyCommand(RegisteredCommand):
    command_name = "multiply"

    def execute(self, a, b):
        return a * b


In [67]:
RegistryMeta.registry


{'add': __main__.AddCommand, 'multiply': __main__.MultiplyCommand}

Now we can create command objects by string name.


In [68]:
add = RegisteredCommand.create("add")
multiply = RegisteredCommand.create("multiply")


In [69]:
add.execute(3, 4), multiply.execute(3, 4)


(7, 12)

In [70]:
assert add.execute(3, 4) == 7
assert multiply.execute(3, 4) == 12


Let's also verify that duplicate registration is rejected.


In [71]:
try:
    class AnotherAddCommand(RegisteredCommand):
        command_name = "add"
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: duplicate command_name: add


### What happened?

Every time a subclass was defined, Python called our metaclass.

The metaclass created the class and then inserted that class object into the registry.

So the registry contains actual classes, not instances.


# Problem 9 — Understand Metaclass `__call__`

So far we have focused on creating **classes**.

But after a class exists, what happens when we do this?

```python
obj = SomeClass(...)
```

The class object is callable.

And the method responsible for that call lives on the **metaclass**.


Let's create a metaclass that prints the steps.


In [72]:
class TraceCallMeta(type):
    def __call__(cls, *args, **kwargs):
        print(f"metaclass __call__: constructing {cls.__name__}")
        obj = super().__call__(*args, **kwargs)
        print(f"metaclass __call__: finished {cls.__name__}")
        return obj


Now define an ordinary class using that metaclass.

We will also print from the instance-level `__new__` and `__init__`.


In [73]:
class TracedObject(metaclass=TraceCallMeta):
    def __new__(cls, value):
        print("instance __new__")
        return super().__new__(cls)

    def __init__(self, value):
        print("instance __init__")
        self.value = value


Now create an instance and watch the order.


In [74]:
traced = TracedObject(100)


metaclass __call__: constructing TracedObject
instance __new__
instance __init__
metaclass __call__: finished TracedObject


The rough call chain is:

1. metaclass `__call__`
2. class `__new__`
3. class `__init__`
4. metaclass `__call__` returns the finished object


In [75]:
traced.value


100

In [76]:
assert traced.value == 100


This distinction is extremely important:

- metaclass `__new__` creates **class objects**
- class `__new__` creates **instances**
- metaclass `__call__` controls what happens when the class is called


# Problem 10 — Use Metaclass `__call__` for Instance Caching

Now that we understand metaclass `__call__`, let's use it for something practical.

We want a `Token` class such that:

```python
Token("python") is Token("python")
```

is `True`.

In other words, repeated construction with the same name should return the same object.


The cache should belong to each class using the metaclass.

So let's create it when the class itself is created.


In [77]:
class InternMeta(type):
    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        cls._instance_cache = {}
        return cls


Now override `__call__`.

This is the hook that runs when we write `Token(...)`.


In [78]:
class InternMeta(InternMeta):
    def __call__(cls, name):
        if name not in cls._instance_cache:
            cls._instance_cache[name] = super().__call__(name)

        return cls._instance_cache[name]


Now define the instance class itself.


In [79]:
class Token(metaclass=InternMeta):
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Token({self.name!r})"


Create two tokens with the same input.


In [80]:
a = Token("python")
b = Token("python")


They should be the same object.


In [81]:
a is b


True

Now use a different name.


In [82]:
c_token = Token("java")
a is c_token


False

In [83]:
assert a is b
assert a is not c_token


Let's inspect the cache.


In [84]:
Token._instance_cache


{'python': Token('python'), 'java': Token('java')}

### Important limitation

This simple cache only accepts one hashable name.

A more general implementation would need to decide how positional and keyword arguments are normalized into cache keys.


# Problem 11 — A Descriptor Created by a Metaclass

Let's combine class construction with descriptors.

Suppose a class defines annotations like this:

```python
class Product:
    name: str
    quantity: int
```

We want a metaclass to replace those annotated names with validating descriptors automatically.


First we need a descriptor.

The descriptor will know:

- its name
- the expected type


In [85]:
class TypedField:
    def __init__(self, expected_type):
        self.expected_type = expected_type
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self

        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        if not isinstance(value, self.expected_type):
            raise TypeError(
                f"{self.name} must be {self.expected_type.__name__}"
            )

        instance.__dict__[self.name] = value


Now let's write the metaclass.

It will inspect the class annotations before the class is constructed.


In [86]:
class TypedMeta(type):
    def __new__(mcls, name, bases, namespace):
        annotations = namespace.get("__annotations__", {})

        for field_name, expected_type in annotations.items():
            if field_name not in namespace:
                namespace[field_name] = TypedField(expected_type)

        return super().__new__(mcls, name, bases, namespace)


Notice an important detail.

We insert the descriptors **before** calling `type.__new__`.

That allows normal descriptor initialization, including `__set_name__`, to happen during class creation.


Let's create a class.


In [87]:
class TypedProduct(metaclass=TypedMeta):
    name: str
    quantity: int

    def __init__(self, name, quantity):
        self.name = name
        self.quantity = quantity


Let's inspect the class namespace.


In [88]:
TypedProduct.__dict__["name"], TypedProduct.__dict__["quantity"]


(<__main__.TypedField at 0x2392a3ef4d0>,
 <__main__.TypedField at 0x2392a48c2d0>)

Those names are now descriptors.


In [89]:
isinstance(TypedProduct.__dict__["name"], TypedField)


True

Create a valid instance.


In [90]:
product = TypedProduct("Keyboard", 5)
product.name, product.quantity


('Keyboard', 5)

Now try an invalid assignment.


In [91]:
try:
    product.quantity = "five"
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: quantity must be int


In [92]:
assert product.name == "Keyboard"
assert product.quantity == 5


### Why this pattern is useful

A class body can remain compact and declarative while a metaclass turns that declaration into richer runtime behavior.

Frameworks often use related ideas for models, schemas, ORM fields and validation.


# Problem 12 — Preserve Declaration Order and Build a Serializer

Let's continue with fields, but now declaration order matters.

Suppose we write:

```python
class Point:
    x = Field()
    y = Field()
    label = Field()
```

We want serialization to preserve exactly that order.


Modern Python dictionaries preserve insertion order.

But using `__prepare__` makes the intent explicit and lets us control the namespace if we need more behavior later.


In [93]:
class Field:
    def __init__(self):
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self

        return instance.__dict__[self.name]

    def __set__(self, instance, value):
        instance.__dict__[self.name] = value


Now let's create the metaclass.


In [94]:
class SerializableMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases):
        return OrderedDict()

    def __new__(mcls, name, bases, namespace):
        inherited_fields = []

        for base in bases:
            inherited_fields.extend(
                getattr(base, "__field_names__", ())
            )

        local_fields = [
            key
            for key, value in namespace.items()
            if isinstance(value, Field)
        ]

        namespace["__field_names__"] = tuple(
            inherited_fields + local_fields
        )

        return super().__new__(
            mcls,
            name,
            bases,
            dict(namespace),
        )


Now create a base class that knows how to serialize itself using the generated field list.


In [95]:
class Serializable(metaclass=SerializableMeta):
    def to_tuple(self):
        return tuple(
            getattr(self, name)
            for name in self.__field_names__
        )


Now a concrete class.


In [96]:
class OrderedPoint(Serializable):
    x = Field()
    y = Field()
    label = Field()

    def __init__(self, x, y, label):
        self.x = x
        self.y = y
        self.label = label


Let's inspect the field order captured by the metaclass.


In [97]:
OrderedPoint.__field_names__


('x', 'y', 'label')

And serialize an instance.


In [98]:
point = OrderedPoint(10, 20, "A")
point.to_tuple()


(10, 20, 'A')

In [99]:
assert OrderedPoint.__field_names__ == ("x", "y", "label")
assert point.to_tuple() == (10, 20, "A")


Now let's verify inherited fields are preserved too.


In [100]:
class ColoredPoint(OrderedPoint):
    color = Field()

    def __init__(self, x, y, label, color):
        super().__init__(x, y, label)
        self.color = color


In [101]:
ColoredPoint.__field_names__


('x', 'y', 'label', 'color')

In [102]:
colored = ColoredPoint(1, 2, "P", "blue")
colored.to_tuple()


(1, 2, 'P', 'blue')

In [103]:
assert ColoredPoint.__field_names__ == (
    "x",
    "y",
    "label",
    "color",
)


### A subtle design point

When a metaclass accumulates information from base classes, you must decide whether subclasses:

- append
- override
- merge
- replace

There is no universal answer.

That behavior is part of the framework design.


# Problem 13 — A Metaclass Conflict

Let's look at a problem that appears when multiple inheritance meets metaclasses.

We will create two unrelated metaclasses.


In [104]:
class MetaLeft(type):
    pass

class MetaRight(type):
    pass


Now two classes, each using one of them.


In [105]:
class Left(metaclass=MetaLeft):
    pass

class Right(metaclass=MetaRight):
    pass


Now suppose we want this:

```python
class Combined(Left, Right):
    pass
```

Python needs one metaclass for `Combined`.

That metaclass must be compatible with the metaclasses of both bases.


Let's try the naive version.


In [106]:
try:
    class BrokenCombined(Left, Right):
        pass
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases


The problem is that `MetaLeft` and `MetaRight` are unrelated.

Neither one is a subclass of the other.


A common resolution is to define a combined metaclass.


In [107]:
class CombinedMeta(MetaLeft, MetaRight):
    pass


Now explicitly use that metaclass.


In [108]:
class Combined(Left, Right, metaclass=CombinedMeta):
    pass


Let's inspect the result.


In [109]:
type(Combined), Combined.__mro__


(__main__.CombinedMeta,
 (__main__.Combined, __main__.Left, __main__.Right, object))

In [110]:
assert isinstance(Combined, MetaLeft)
assert isinstance(Combined, MetaRight)


### Why this works

`CombinedMeta` is a subclass of both `MetaLeft` and `MetaRight`.

So it satisfies the metaclass requirements imposed by both base classes.


# Problem 14 — Why `__classcell__` Can Matter

This is one of the more advanced class-construction details.

Functions that use zero-argument `super()` need access to the class they belong to.

Python supports this using a special mechanism involving `__class__` and, during class creation, `__classcell__`.


We will first define a normal class that uses zero-argument `super()`.


In [111]:
class Parent:
    def greet(self):
        return "parent"

class Child(Parent):
    def greet(self):
        return super().greet() + " -> child"


The method works normally.


In [112]:
Child().greet()


'parent -> child'

Let's inspect the function's closure variables.


In [113]:
inspect.getclosurevars(Child.__dict__["greet"])


ClosureVars(nonlocals={'__class__': <class '__main__.Child'>}, globals={}, builtins={'super': <class 'super'>}, unbound=set())

The function has access to `__class__`.

That is what allows zero-argument `super()` to determine which class the method was defined in.


This is a useful warning when manually manipulating class namespaces in custom metaclasses:

if Python places `__classcell__` in the namespace, a metaclass should generally pass it through to `type.__new__` rather than silently discarding it.


Let's create a metaclass that simply records whether the namespace contained `__classcell__`.


In [114]:
class ClassCellMeta(type):
    def __new__(mcls, name, bases, namespace):
        print(
            name,
            "contains __classcell__:",
            "__classcell__" in namespace,
        )

        return super().__new__(
            mcls,
            name,
            bases,
            namespace,
        )


Now define one class with no zero-argument `super()`.


In [115]:
class NoSuper(metaclass=ClassCellMeta):
    def method(self):
        return 1


NoSuper contains __classcell__: False


And one that uses zero-argument `super()`.


In [116]:
class BaseForSuper:
    def method(self):
        return "base"

class UsesSuper(BaseForSuper, metaclass=ClassCellMeta):
    def method(self):
        return super().method() + " + subclass"


UsesSuper contains __classcell__: True


In [117]:
UsesSuper().method()


'base + subclass'

In [118]:
assert UsesSuper().method() == "base + subclass"


### Best practice

If you override metaclass `__new__`, do not rebuild the namespace carelessly.

Preserve interpreter-generated entries unless you have a very specific reason not to.


# Problem 15 — Build a Small Class Factory Step by Step

For the final problem we will build a function that creates record-like classes.

We want to be able to write:

```python
Point = make_record("Point", ["x", "y"])
p = Point(10, 20)
```

and get:

```python
repr(p)
# "Point(x=10, y=20)"
```

We also want equality between objects of the same generated type.


Let's not write the whole factory at once.

First, think about what information the factory receives.


In [119]:
record_name = "Point"
record_fields = ("x", "y")


The generated `__init__` needs access to `record_fields`.

So we can create it as a closure.


In [120]:
def make_init(fields):
    def __init__(self, *values):
        if len(values) != len(fields):
            raise TypeError(
                f"expected {len(fields)} values, got {len(values)}"
            )

        for field, value in zip(fields, values):
            setattr(self, field, value)

    return __init__


Let's generate the function.


In [121]:
point_init = make_init(record_fields)


The representation function also needs the field names.


In [122]:
def make_repr(class_name, fields):
    def __repr__(self):
        pairs = ", ".join(
            f"{field}={getattr(self, field)!r}"
            for field in fields
        )

        return f"{class_name}({pairs})"

    return __repr__


In [123]:
point_repr = make_repr(record_name, record_fields)


Now equality.

We want two objects to compare equal only if:

1. they have exactly the same generated class
2. every field has the same value


In [124]:
def make_eq(fields):
    def __eq__(self, other):
        if type(other) is not type(self):
            return NotImplemented

        return all(
            getattr(self, field) == getattr(other, field)
            for field in fields
        )

    return __eq__


Now we can assemble the namespace.


In [125]:
record_namespace = {
    "_fields": record_fields,
    "__init__": point_init,
    "__repr__": point_repr,
    "__eq__": make_eq(record_fields),
}


Let's inspect it before creating the class.


In [126]:
record_namespace


{'_fields': ('x', 'y'),
 '__init__': <function __main__.make_init.<locals>.__init__(self, *values)>,
 '__repr__': <function __main__.make_repr.<locals>.__repr__(self)>,
 '__eq__': <function __main__.make_eq.<locals>.__eq__(self, other)>}

Now create the class.


In [127]:
Point = type(
    record_name,
    (),
    record_namespace,
)


Let's use it.


In [128]:
p1 = Point(10, 20)
p2 = Point(10, 20)
p3 = Point(20, 10)


In [129]:
p1


Point(x=10, y=20)

In [130]:
p1 == p2, p1 == p3


(True, False)

In [131]:
assert repr(p1) == "Point(x=10, y=20)"
assert p1 == p2
assert p1 != p3


Now that the individual pieces work, we can package them into a reusable factory.


In [132]:
def make_record(class_name, fields):
    fields = tuple(fields)

    if not fields:
        raise ValueError("at least one field is required")

    if len(set(fields)) != len(fields):
        raise ValueError("field names must be unique")

    for field in fields:
        if not isinstance(field, str) or not field.isidentifier():
            raise ValueError(
                f"invalid field name: {field!r}"
            )

    namespace = {
        "_fields": fields,
        "__init__": make_init(fields),
        "__repr__": make_repr(class_name, fields),
        "__eq__": make_eq(fields),
    }

    return type(
        class_name,
        (),
        namespace,
    )


Now let's create a completely different class from the same factory.


In [133]:
Book = make_record(
    "Book",
    ["title", "author", "year"],
)


In [134]:
book = Book(
    "The Python Journey",
    "A. Programmer",
    2026,
)


In [135]:
book


Book(title='The Python Journey', author='A. Programmer', year=2026)

In [136]:
Book._fields


('title', 'author', 'year')

In [137]:
assert repr(book) == (
    "Book(title='The Python Journey', "
    "author='A. Programmer', year=2026)"
)


Let's also test validation in the factory.


In [138]:
try:
    make_record("Broken", ["x", "x"])
except ValueError as exc:
    print(type(exc).__name__ + ":", exc)


ValueError: field names must be unique


### Final lesson from the factory

Dynamic classes are most useful when the class structure genuinely comes from runtime data or a declarative specification.

For ordinary classes, normal `class` syntax is usually clearer.

Metaprogramming is powerful precisely because it should solve problems that ordinary class definitions cannot solve cleanly.


# Additional Practice Problems

The following problems are shorter.

Try to solve each one before looking at the solution immediately below it.


## Practice Problem A — Add Methods After Class Creation

Create an empty class dynamically.

Then attach a method `double(self, value)` using `setattr`.

Finally verify that existing instances can use the newly attached method.


First create the class.


In [139]:
DynamicEmpty = type("DynamicEmpty", (), {})
instance = DynamicEmpty()


Now define a normal function.


In [140]:
def double(self, value):
    return value * 2


Attach it to the class.


In [141]:
setattr(DynamicEmpty, "double", double)


The existing instance now sees the method through normal attribute lookup.


In [142]:
instance.double(21)


42

In [143]:
assert instance.double(21) == 42


This works because methods are resolved from the class when the attribute is accessed.

The method did not need to exist when the instance was originally created.


## Practice Problem B — Inspect a Class Dictionary Safely

What type is `SomeClass.__dict__`?

Can we write directly into it?


In [144]:
class DictionaryExample:
    x = 10


In [145]:
type(DictionaryExample.__dict__)


mappingproxy

The object is a `mappingproxy`.

It provides a read-only view of the class namespace.


In [146]:
try:
    DictionaryExample.__dict__["y"] = 20
except TypeError as exc:
    print(type(exc).__name__ + ":", exc)


TypeError: 'mappingproxy' object does not support item assignment


But we can still modify the class itself using `setattr`.


In [147]:
setattr(DictionaryExample, "y", 20)
DictionaryExample.y


20

In [148]:
assert DictionaryExample.y == 20


## Practice Problem C — Which Object Is an Instance of Which Class?

Let's make the relationships explicit.


In [149]:
class Sample:
    pass

sample = Sample()


`sample` is an instance of `Sample`.


In [150]:
type(sample) is Sample


True

`Sample` is itself an instance of `type`.


In [151]:
type(Sample) is type


True

And `type` is itself an instance of `type`.


In [152]:
type(type) is type


True

In [153]:
assert type(sample) is Sample
assert type(Sample) is type
assert type(type) is type


This self-referential relationship is one of the reasons Python's object model can feel strange at first.

But it also gives Python a very uniform model: classes are objects too.


# Summary

We covered quite a lot of machinery.

Let's collect the main ideas.


A normal class definition can be thought of in broad steps:

1. Python determines the appropriate metaclass.
2. The metaclass may provide a namespace through `__prepare__`.
3. The class body executes in that namespace.
4. The namespace now contains the attributes and functions created by the body.
5. The metaclass creates the class object.
6. Later, calling the class usually invokes metaclass `__call__`, which coordinates instance construction.


We also saw that these stages give us different extension points.

`__prepare__` is useful when we want to control or observe the class namespace while the body executes.

Metaclass `__new__` is useful when we want to inspect, validate or transform the namespace before the finished class is returned.

Metaclass `__call__` is useful when we want to control what happens when the finished class is called to create or retrieve instances.


We also combined this machinery with:

- closures
- inheritance
- multiple inheritance
- descriptors
- registration systems
- annotations
- dynamic class factories
- instance caching
- class-body validation


The main best practice is still simple:

**Use the least powerful mechanism that solves the problem clearly.**

If normal class syntax works well, use it.

If a class decorator is enough, prefer that over a metaclass.

Use metaclasses when the problem genuinely concerns class construction itself.


# End

A good next exercise is to take one of the metaclasses in this notebook and rewrite it as a class decorator.

Then compare:

- what information is available
- when the transformation runs
- what can and cannot be changed

That comparison makes the purpose of metaclasses much clearer.
